# 05 — Ablations, image-fallback variant, cross-dataset tests, bootstrap statistics
**Requires:** notebooks 01–04 finished. Kernel **Python (internship1)**.

| Part | What | Seeds |
|---|---|---|
| A | **NAFIM + image fallback** — final prediction = r · p(multimodal) + (1 − r) · p(image only) | 5 |
| B | **Cross-dataset (binary harmful)** — train on BHM or MIMOSA, test on BHM / MIMOSA / BanglaAbuseMeme with **OCR text** | 3 |
| C | **Ablations** — NAFIM without gate / reliability loss / consistency loss / synthetic noise | 3 |
| D | **Final model selection by validation F1** (NAFIM vs. NAFIM + image fallback) — never by test F1 | — |
| E | **Paired bootstrap** over test memes (pooled across seeds) — more statistical power than a t-test over 5 seeds | — |

**Runtime:** ~6–7 h for all parts, resumable (order: A → B → C, so the most important results arrive first).
Lower `ABL_SEEDS` / `XD_SEEDS` to `[42]` for a quick check.

## 1. Imports & config

In [ ]:
import os, re, gc, json, time, random, platform, unicodedata
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before importing torch
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
import transformers, sklearn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score
from scipy.stats import ttest_rel
from rapidfuzz.distance import Levenshtein
import matplotlib.pyplot as plt

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P, FEAT = f"{ROOT}/processed", f"{ROOT}/features"
NB02, NB03, NB04 = (f"{ROOT}/results/{x}" for x in ["02_finetuned", "03_ocr_eval", "04_nafim"])
RES = f"{ROOT}/results/05_ablation_crossdataset"
PRED = f"{RES}/preds"
os.makedirs(PRED, exist_ok=True)

DATASETS = ["bhm", "mimosa"]                         # datasets with gold captions (used for training)
TASKS = [("bhm", "label"), ("mimosa", "label"), ("mimosa", "label_bin")]
XD_TRAIN = [("bhm", "label"), ("mimosa", "label_bin")]  # binary tasks used as cross-dataset sources
XD_TARGETS = {"bhm": "text_ocr", "mimosa": "text_ocr", "banglaabusememe": "text"}  # OCR text everywhere
SEEDS = [42, 43, 44, 45, 46]
ABL_SEEDS = [42, 43, 44]
XD_SEEDS = [42, 43, 44]
TEXT_MODELS = {"BanglaBERT": "csebuetnlp/banglabert", "XLM-R": "xlm-roberta-base"}

REAL_SOURCES = {"gold": "text", "surya": "text_surya", "easyocr": "text_easyocr", "ocr": "text_ocr"}
SYNTH_K, SYNTH_SEED = 3, 0
EVAL_SOURCES = ["gold", "surya", "easyocr"]
HIGH_CER = 0.5
LAMBDA_IMG = 0.5            # auxiliary loss on the image-only expert (image-fallback variant)
N_BOOT = 2000
EVAL_BS = 64

dev = "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| AMP:", AMP_DTYPE)
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 2**30:.2f} / {total / 2**30:.2f} GiB  (close other GPU programs if below ~6 GiB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def slug(s):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", s).strip("_")

## 2. Settings inherited from notebooks 02–04

In [ ]:
cfg02 = json.load(open(f"{NB02}/config.json"))["text"]
cfg03 = json.load(open(f"{NB03}/config.json"))
cfg04 = json.load(open(f"{NB04}/config.json"))
MAX_LEN, BS = cfg02["max_len"], cfg02["batch_size"]
MAX_EPOCHS, PATIENCE = cfg02["max_epochs"], cfg02["patience"]
LR_ENC, LR_HEAD, WD = cfg02["lr_encoder"], cfg02["lr_head"], cfg02["weight_decay"]
WARMUP, DROPOUT = cfg02["warmup"], cfg02["dropout"]
BASE_MODEL = cfg03["selected_model"]
ENC_KEY = BASE_MODEL.split(" ", 1)[1].replace("+CLIP", "")
FREEZE_WORD_EMB = bool(cfg04["word_embeddings_frozen"])
LAMBDA_REL, LAMBDA_CONS = cfg04["nafim"]["lambda_rel"], cfg04["nafim"]["lambda_cons"]
NAFIM_MIX = cfg04["nafim"]["text_source_mix"]
MIXED_MIX = cfg04["mixed_baseline"]["text_source_mix"]
NO_SYNTH_MIX = {"gold": 1 / 3, "surya": 1 / 3, "easyocr": 1 / 3}
print("encoder:", ENC_KEY, "| frozen word emb:", FREEZE_WORD_EMB, "| lambda_rel:", LAMBDA_REL, "| lambda_cons:", LAMBDA_CONS)
print("NAFIM mix:", NAFIM_MIX)

# every model variant trained in this notebook
VARIANTS = {
    "Fusion (mixed text)":        dict(nafim=False, gate=False, lrel=0.0,        lcons=0.0,         mix=MIXED_MIX,    fallback=False),
    "NAFIM":                      dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=False),
    "NAFIM + image fallback":     dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=True),
    "NAFIM w/o gate":             dict(nafim=True,  gate=False, lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=False),
    "NAFIM w/o reliability loss": dict(nafim=True,  gate=True,  lrel=0.0,        lcons=LAMBDA_CONS, mix=NAFIM_MIX,    fallback=False),
    "NAFIM w/o consistency":      dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=0.0,         mix=NAFIM_MIX,    fallback=False),
    "NAFIM w/o synthetic noise":  dict(nafim=True,  gate=True,  lrel=LAMBDA_REL, lcons=LAMBDA_CONS, mix=NO_SYNTH_MIX, fallback=False),
}
ABLATIONS = ["NAFIM w/o gate", "NAFIM w/o reliability loss", "NAFIM w/o consistency", "NAFIM w/o synthetic noise"]

## 3. Data: text sources, synthetic noise, CER (identical to notebook 04) + BanglaAbuseMeme for cross-dataset tests

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[^\u0980-\u09FFa-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def cer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    return np.nan if not r else Levenshtein.distance(r, h) / len(r)

BN_LETTERS = [chr(c) for c in range(0x0985, 0x09BA)]
BN_SIGNS = list("\u09BE\u09BF\u09C0\u09C1\u09C2\u09C3\u09C7\u09C8\u09CB\u09CC\u09CD\u0982\u0983\u0981")
LATIN = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

def corrupt(text, rng):
    rate = rng.uniform(0.05, 0.45)
    out = []
    for ch in str(text):
        u = rng.random()
        if u < rate * 0.30:
            continue
        elif u < rate * 0.60:
            out.append(rng.choice(BN_LETTERS + LATIN))
        elif u < rate * 0.75:
            out.append(ch); out.append(rng.choice(BN_LETTERS + LATIN))
        elif u < rate and ch in BN_SIGNS:
            continue
        else:
            out.append(ch)
    words = "".join(out).split()
    if words and rng.random() < 0.2:
        words.pop(rng.randrange(len(words)))
    if rng.random() < 0.3:
        junk = "".join(rng.choice(LATIN) for _ in range(rng.randint(3, 8)))
        words.insert(rng.randint(0, len(words)), junk)
    return " ".join(words)

def idx_of(df, s):
    return np.where(df["split"].values == s)[0]

data = {}
for ds in DATASETS:
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    for c in REAL_SOURCES.values():
        df[c] = df[c].astype(str)
    rng = random.Random(f"{SYNTH_SEED}-{ds}")
    for k in range(SYNTH_K):
        df[f"text_synth{k}"] = [corrupt(t, rng) for t in df["text"]]
    img = np.load(f"{FEAT}/{ds}_img.npy").astype(np.float32)
    assert len(df) == len(img)
    data[ds] = {"df": df, "img": torch.from_numpy(img)}

SOURCES = list(REAL_SOURCES) + [f"synth{k}" for k in range(SYNTH_K)]
SRC_COL = {**REAL_SOURCES, **{f"synth{k}": f"text_synth{k}" for k in range(SYNTH_K)}}
SRC_IDX = {s: i for i, s in enumerate(SOURCES)}
for ds in DATASETS:
    df = data[ds]["df"]
    for s in SOURCES:
        df[f"cer_{s}"] = 0.0 if s == "gold" else [cer(r, h) for r, h in zip(df["text"], df[SRC_COL[s]])]
    df.loc[df["text"].map(norm) == "", [f"cer_{s}" for s in SOURCES]] = np.nan

bam = pd.read_csv(f"{P}/banglaabusememe.csv").fillna("")
bam["text"] = bam["text"].astype(str)
bam_img = np.load(f"{FEAT}/banglaabusememe_img.npy").astype(np.float32)
assert len(bam) == len(bam_img)
print("BanglaAbuseMeme:", bam.shape, "| test memes:", (bam["split"] == "test").sum())

## 4. Tokenization, text-quality features, reliability targets, cross-dataset test tensors

In [ ]:
tok = AutoTokenizer.from_pretrained(TEXT_MODELS[ENC_KEY])

def quality_features(texts):
    pieces = [len(x) for x in tok(list(texts), add_special_tokens=False)["input_ids"]]
    rows = []
    for t, npc in zip(texts, pieces):
        t = str(t); n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        bn = sum("\u0980" <= c <= "\u09FF" for c in t); lat = sum(c.isascii() and c.isalpha() for c in t)
        dig = sum(c.isdigit() for c in t); sp = sum(c.isspace() for c in t)
        pun = sum(c in ".,!?;:'\"-।()" for c in t)
        rows.append([np.log1p(len(t)), np.log1p(len(words)), float(len(t.strip()) == 0),
                     bn / n, lat / n, dig / n, max(n - bn - lat - dig - sp - pun, 0) / n, pun / n,
                     sum(len(w) == 1 for w in words) / nw, np.mean([len(w) for w in words]) if words else 0.0,
                     npc / nw])
    return np.array(rows, dtype=np.float32)

def encode(texts):
    e = tok(list(texts), padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return e["input_ids"], e["attention_mask"]

for ds in DATASETS:
    df = data[ds]["df"]
    enc = [encode(df[SRC_COL[s]]) for s in SOURCES]
    data[ds]["ids"] = torch.stack([e[0] for e in enc]); data[ds]["mask"] = torch.stack([e[1] for e in enc])
    Q = np.stack([quality_features(df[SRC_COL[s]].tolist()) for s in SOURCES])
    tr = idx_of(df, "train")
    train_src = [SRC_IDX[s] for s in ["gold", "surya", "easyocr"]] + [SRC_IDX[f"synth{k}"] for k in range(SYNTH_K)]
    mu = Q[train_src][:, tr].reshape(-1, Q.shape[-1]).mean(0); sd = Q[train_src][:, tr].reshape(-1, Q.shape[-1]).std(0) + 1e-6
    data[ds]["q_mu"], data[ds]["q_sd"] = mu, sd
    data[ds]["q"] = torch.from_numpy((Q - mu) / sd)
    rel = np.stack([np.clip(1 - df[f"cer_{s}"].values, 0, 1) for s in SOURCES])
    data[ds]["rel"] = torch.from_numpy(rel.astype(np.float32))
    print(ds, "ids", tuple(data[ds]["ids"].shape))

# cross-dataset test sets (OCR text, binary label, test split); quality features are standardised later
# with the statistics of the TRAINING dataset
xd = {}
for tgt, col in XD_TARGETS.items():
    df = bam if tgt == "banglaabusememe" else data[tgt]["df"]
    te = idx_of(df, "test"); texts = df[col].astype(str).values[te]
    ids, mask = encode(texts)
    img = torch.from_numpy(bam_img[te]) if tgt == "banglaabusememe" else data[tgt]["img"][te]
    xd[tgt] = {"ids": ids, "mask": mask, "img": img, "Qraw": quality_features(texts),
               "y": df["label_bin"].values[te].astype(int)}
    print("cross-dataset target", tgt, "| test memes:", len(te))

for ds, lab in TASKS:
    df = data[ds]["df"]
    np.save(f"{PRED}/{ds}__{lab}__ytrue.npy", df[lab].values[idx_of(df, "test")].astype(int))

## 5. Models (NAFIM with optional gate and optional image-only fallback expert)

In [ ]:
def build_encoder():
    enc = AutoModel.from_pretrained(TEXT_MODELS[ENC_KEY])
    if FREEZE_WORD_EMB:
        enc.get_input_embeddings().weight.requires_grad_(False)
    return enc

def mean_pool(hs, mask):
    m = mask.unsqueeze(-1).to(hs.dtype)
    return (hs * m).sum(1) / m.sum(1).clamp(min=1)

class FusionBaseline(nn.Module):
    def __init__(self, n, img_dim):
        super().__init__()
        self.enc = build_encoder(); h = self.enc.config.hidden_size
        self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * h, h), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
        self.img_logits = None
    def forward(self, ids, mask, img, q=None):
        t = mean_pool(self.enc(input_ids=ids, attention_mask=mask).last_hidden_state, mask)
        return self.head(torch.cat([t, self.img_proj(img)], -1)), None

class NAFIM(nn.Module):
    """Reliability-gated fusion. With fallback=True the output is a reliability-weighted mixture
    r * p(multimodal) + (1 - r) * p(image only); log-probabilities are returned, so softmax/CE stay valid."""
    def __init__(self, n, img_dim, q_dim, gate=True, fallback=False):
        super().__init__()
        self.enc = build_encoder(); h = self.enc.config.hidden_size
        self.gate, self.fallback = gate, fallback
        self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.rel_head = nn.Sequential(nn.Linear(h + q_dim, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * h + 1, h), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
        if fallback:
            self.img_head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(h, n))
        self.img_logits = None
    def forward(self, ids, mask, img, q):
        t = mean_pool(self.enc(input_ids=ids, attention_mask=mask).last_hidden_state, mask)
        r = torch.sigmoid(self.rel_head(torch.cat([t, q.to(t.dtype)], -1)))
        tg = r * t if self.gate else t
        iv = self.img_proj(img)
        lo = self.head(torch.cat([tg, iv, r], -1))
        if not self.fallback:
            return lo, r.squeeze(-1)
        li = self.img_head(iv)
        self.img_logits = li
        rf = r.float()
        p = rf * F.softmax(lo.float(), -1) + (1 - rf) * F.softmax(li.float(), -1)
        return torch.log(p.clamp(min=1e-8)), r.squeeze(-1)

## 6. Training (same procedure as notebook 04, configurable per variant)

In [ ]:
def class_weights(y, n):
    counts = np.bincount(y, minlength=n)
    return torch.tensor(len(y) / (n * np.maximum(counts, 1)), dtype=torch.float32, device=dev)

def sample_sources(n_items, mix, g):
    groups = list(mix)
    pick = torch.multinomial(torch.tensor([mix[k] for k in groups]), n_items, replacement=True, generator=g)
    synth = torch.randint(0, SYNTH_K, (n_items,), generator=g)
    return torch.tensor([SRC_IDX[f"synth{int(synth[i])}"] if groups[p] == "synth" else SRC_IDX[groups[p]]
                         for i, p in enumerate(pick.tolist())])

@torch.no_grad()
def _predict(model, ids, mask, img, q):
    model.eval(); probs, rels = [], []
    for i in range(0, len(ids), EVAL_BS):
        sl = slice(i, i + EVAL_BS)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            lo, r = model(ids[sl].to(dev), mask[sl].to(dev), img[sl].to(dev), q[sl].to(dev))
        probs.append(torch.softmax(lo.float(), 1).cpu())
        rels.append(r.float().cpu() if r is not None else torch.full((len(ids[sl]),), float("nan")))
    return torch.cat(probs).numpy(), torch.cat(rels).numpy()

def predict(model, ds, src, idx):
    d = data[ds]; s = SRC_IDX[src]; b = torch.from_numpy(idx)
    return _predict(model, d["ids"][s, b], d["mask"][s, b], d["img"][b], d["q"][s, b])

def predict_xd(model, src_ds, tgt):
    t = xd[tgt]
    q = torch.from_numpy(((t["Qraw"] - data[src_ds]["q_mu"]) / data[src_ds]["q_sd"]).astype(np.float32))
    return _predict(model, t["ids"], t["mask"], t["img"], q)

def run_train(name, ds, lab, seed, ckpt=False, cross=False):
    v = VARIANTS[name]
    set_seed(seed)
    d = data[ds]; df = d["df"]; y = df[lab].values.astype(np.int64); n = int(y.max()) + 1
    Y = torch.from_numpy(y)
    tr, va, te = [idx_of(df, s) for s in ("train", "val", "test")]
    if v["nafim"]:
        model = NAFIM(n, d["img"].shape[1], d["q"].shape[-1], gate=v["gate"], fallback=v["fallback"]).to(dev)
    else:
        model = FusionBaseline(n, d["img"].shape[1]).to(dev)
    if ckpt:
        model.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    enc_ids = {id(p) for p in model.enc.parameters()}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    enc_params = [p for p in model.enc.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENC}, {"params": head_params, "lr": LR_HEAD}],
                            weight_decay=WD, foreach=False)
    spe = (len(tr) + BS - 1) // BS
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * spe * MAX_EPOCHS), spe * MAX_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    cw = class_weights(y[tr], n)
    g = torch.Generator().manual_seed(seed)
    trainable = [p for p in model.parameters() if p.requires_grad]

    best_f, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.from_numpy(tr)[torch.randperm(len(tr), generator=g)]
        for i in tqdm(range(0, len(perm), BS), desc=f"{slug(name)} {ds}/{lab} s{seed} ep{ep + 1}", leave=False):
            b = perm[i:i + BS]; src = sample_sources(len(b), v["mix"], g); yb = Y[b].to(dev)
            if v["nafim"]:
                gold = torch.full_like(src, SRC_IDX["gold"])
                s2, b2 = torch.cat([gold, src]), torch.cat([b, b])
                ids, mask = d["ids"][s2, b2].to(dev), d["mask"][s2, b2].to(dev)
                img, q, rt = d["img"][b2].to(dev), d["q"][s2, b2].to(dev), d["rel"][s2, b2].to(dev)
            else:
                ids, mask = d["ids"][src, b].to(dev), d["mask"][src, b].to(dev)
                img, q = d["img"][b].to(dev), d["q"][src, b].to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                lo, r = model(ids, mask, img, q)
            lo = lo.float()
            if v["nafim"]:
                lg, lv = lo[:len(b)], lo[len(b):]
                yy = torch.cat([yb, yb])
                loss = 0.5 * (F.cross_entropy(lg, yb, weight=cw) + F.cross_entropy(lv, yb, weight=cw))
                if v["lrel"] > 0:
                    ok = ~torch.isnan(rt)
                    if ok.any():
                        loss = loss + v["lrel"] * F.mse_loss(r.float()[ok], rt[ok])
                if v["lcons"] > 0:
                    loss = loss + v["lcons"] * F.kl_div(F.log_softmax(lv, -1), F.softmax(lg, -1).detach(), reduction="batchmean")
                if v["fallback"]:
                    loss = loss + LAMBDA_IMG * F.cross_entropy(model.img_logits.float(), yy, weight=cw)
            else:
                loss = F.cross_entropy(lo, yb, weight=cw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        f = np.mean([f1_score(y[va], predict(model, ds, s, va)[0].argmax(1), average="macro") for s in EVAL_SOURCES])
        if f > best_f:
            best_f, best_ep, bad = f, ep + 1, 0
            best_state = {k: v_.detach().cpu().clone() for k, v_ in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    out = {s: predict(model, ds, s, te) for s in REAL_SOURCES}
    xout = {tgt: predict_xd(model, ds, tgt) for tgt in xd if tgt != ds} if cross else {}
    del model, opt, best_state, sch, scaler
    gc.collect(); torch.cuda.empty_cache()
    return best_f, best_ep, out, xout, y[te]

RUNS_CSV = f"{RES}/runs.csv"
done = set(pd.read_csv(RUNS_CSV)["train_id"]) if os.path.exists(RUNS_CSV) else set()

def train_and_log(name, ds, lab, seed, part):
    cross = (ds, lab) in XD_TRAIN
    train_id = f"{ds}__{lab}__{slug(name)}__s{seed}"
    if train_id in done:
        return
    t0 = time.time()
    for ckpt in (False, True):
        try:
            vf, ep, out, xout, y_te = run_train(name, ds, lab, seed, ckpt=ckpt, cross=cross)
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if ckpt:
                raise
            print(f"  out of GPU memory on {train_id} - retrying with gradient checkpointing")
    mins = round((time.time() - t0) / 60, 2)
    rows = []
    for s, (p, r) in out.items():
        np.save(f"{PRED}/{train_id}__test-{s}.npy", p)
        if VARIANTS[name]["nafim"]:
            np.save(f"{PRED}/{train_id}__test-{s}__rel.npy", r)
        rows.append({"train_id": train_id, "part": part, "dataset": ds, "task": lab, "model": name, "seed": seed,
                     "test_on": ds, "test_text": s, "best_epoch": ep, "val_mean_f1": vf,
                     "test_f1": f1_score(y_te, p.argmax(1), average="macro"),
                     "test_acc": accuracy_score(y_te, p.argmax(1)), "minutes": mins})
    for tgt, (p, r) in xout.items():
        np.save(f"{PRED}/{train_id}__xd-{tgt}.npy", p)
        rows.append({"train_id": train_id, "part": part, "dataset": ds, "task": lab, "model": name, "seed": seed,
                     "test_on": tgt, "test_text": "ocr", "best_epoch": ep, "val_mean_f1": vf,
                     "test_f1": f1_score(xd[tgt]["y"], p.argmax(1), average="macro"),
                     "test_acc": accuracy_score(xd[tgt]["y"], p.argmax(1)), "minutes": mins})
    pd.DataFrame(rows).to_csv(RUNS_CSV, mode="a", header=not os.path.exists(RUNS_CSV), index=False)
    done.add(train_id)
    print(f"[{part}] {ds:7s} {lab:10s} {name:28s} s{seed} | " +
          "  ".join(f"{r['test_on'][:6]}/{r['test_text']}={r['test_f1']:.4f}" for r in rows) + f"  ({mins} min)")

## 7. Run all trainings (A → B → C; long, resumable)

In [ ]:
plan = []
plan += [("A", "NAFIM + image fallback", ds, lab, s) for s in SEEDS for ds, lab in TASKS]
plan += [("B", m, ds, lab, s) for s in XD_SEEDS for m in ["Fusion (mixed text)", "NAFIM"] for ds, lab in XD_TRAIN]
plan += [("C", m, ds, lab, s) for s in ABL_SEEDS for m in ABLATIONS for ds, lab in TASKS]
todo = [p for p in plan if f"{p[2]}__{p[3]}__{slug(p[1])}__s{p[4]}" not in done]
print(f"trainings: {len(plan)} planned, {len(plan) - len(todo)} done, {len(todo)} to do")
for part, name, ds, lab, s in todo:
    train_and_log(name, ds, lab, s, part)

## 8. Gather predictions of all models (notebooks 03, 04, 05)

In [ ]:
PRED03, PRED04 = f"{NB03}/preds", f"{NB04}/preds"
BASE_SLUG = BASE_MODEL.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")
def nb04_slug(s):
    return s.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")

def pred_path(model, ds, lab, s, t):
    if model == "Fusion (train gold)":
        return f"{PRED03}/{ds}__{lab}__{BASE_SLUG}__train-gold__s{s}__test-{t}.npy"
    if model == "Fusion (train OCR)":
        return f"{PRED03}/{ds}__{lab}__{BASE_SLUG}__train-ocr__s{s}__test-{t}.npy"
    if model in ("Fusion (mixed text)", "NAFIM"):
        return f"{PRED04}/{ds}__{lab}__{nb04_slug(model)}__s{s}__test-{t}.npy"     # main 5-seed runs from notebook 04
    return f"{PRED}/{ds}__{lab}__{slug(model)}__s{s}__test-{t}.npy"

def load_preds(model, ds, lab, seeds):
    out = {}
    for s in seeds:
        files = {t: pred_path(model, ds, lab, s, t) for t in REAL_SOURCES}
        if all(os.path.exists(f) for f in files.values()):
            out[s] = {t: np.load(f).argmax(1) for t, f in files.items()}
    return out

def hi_mask(ds):
    df = data[ds]["df"]; te = idx_of(df, "test")
    return df["cer_ocr"].values[te] >= HIGH_CER

METRICS = ["gold", "surya", "easyocr", "ocr", "mean(gold,surya,easyocr)", "worst(gold,surya,easyocr)", f"high-CER OCR (≥{HIGH_CER})"]
def metric_rows(model, ds, lab, seeds):
    y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy"); hi = hi_mask(ds); rows = []
    for s, pr in load_preds(model, ds, lab, seeds).items():
        f1 = {t: f1_score(y, pr[t], average="macro") for t in REAL_SOURCES}
        ev = [f1[t] for t in EVAL_SOURCES]
        rows.append({"dataset": ds, "task": lab, "model": model, "seed": s, **f1, METRICS[4]: np.mean(ev),
                     METRICS[5]: np.min(ev), METRICS[6]: f1_score(y[hi], pr["ocr"][hi], average="macro")})
    return rows

def fmt(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else (f"{x.mean():.4f}" if len(x) else "—")

## 9. Part D — choose the final model by VALIDATION F1 (NAFIM vs. NAFIM + image fallback)

In [ ]:
r04 = pd.read_csv(f"{NB04}/runs.csv").drop_duplicates("train_id")
r05 = pd.read_csv(RUNS_CSV).drop_duplicates("train_id") if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=["model"])
val = pd.concat([r04[r04.model == "NAFIM"][["dataset", "task", "seed", "model", "val_mean_f1"]],
                 r05[r05.model == "NAFIM + image fallback"][["dataset", "task", "seed", "model", "val_mean_f1"]]])
val_tab = val.pivot_table(index=["dataset", "task"], columns="model", values="val_mean_f1", aggfunc="mean")
display(val_tab.round(4))
overall = val.groupby("model")["val_mean_f1"].mean()
FINAL = overall.idxmax() if len(overall) == 2 else "NAFIM"
print("Mean validation macro-F1:", overall.round(4).to_dict())
print(">>> FINAL MODEL (chosen on validation only):", FINAL)

## 10. Part A — main comparison including the image-fallback variant (test set, 5 seeds)

In [ ]:
MAIN_MODELS = ["Fusion (train gold)", "Fusion (train OCR)", "Fusion (mixed text)", "NAFIM", "NAFIM + image fallback"]
main_rows = [r for ds, lab in TASKS for m in MAIN_MODELS for r in metric_rows(m, ds, lab, SEEDS)]
main_cmp = pd.DataFrame(main_rows)
main_tab = (main_cmp.groupby(["dataset", "task", "model"])[METRICS].agg(fmt)
            .reindex(pd.MultiIndex.from_tuples([(d, t, m) for d, t in TASKS for m in MAIN_MODELS]), fill_value="—"))
print("seeds per model:", main_cmp.groupby("model")["seed"].nunique().to_dict(), "| final model:", FINAL)
main_tab

## 11. Part E — paired bootstrap: final model vs. each baseline (test memes resampled, seeds pooled)

In [ ]:
def fast_macro_f1(y, p, n):
    cm = np.bincount(n * y + p, minlength=n * n).reshape(n, n)
    tp = np.diag(cm).astype(float); pred = cm.sum(0); true = cm.sum(1)
    present = (pred + true) > 0
    with np.errstate(divide="ignore", invalid="ignore"):
        f1 = np.where(present, 2 * tp / np.maximum(pred + true, 1), 0.0)
    return f1[present].mean()

def boot_metric(pr, y, idx, n, kind, hi_idx=None):
    """pr: {seed: {src: pred}} -> metric averaged over seeds for resampled indices."""
    vals = []
    for s, d in pr.items():
        if kind == "high":
            vals.append(fast_macro_f1(y[hi_idx], d["ocr"][hi_idx], n)); continue
        f = [fast_macro_f1(y[idx], d[t][idx], n) for t in EVAL_SOURCES]
        vals.append(np.mean(f) if kind == "mean" else np.min(f))
    return np.mean(vals)

boot = []
rng = np.random.default_rng(0)
for ds, lab in TASKS:
    y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy"); n = int(y.max()) + 1; N = len(y)
    hi = np.where(hi_mask(ds))[0]
    A = load_preds(FINAL, ds, lab, SEEDS)
    for base in ["Fusion (train gold)", "Fusion (train OCR)", "Fusion (mixed text)"]:
        B = load_preds(base, ds, lab, SEEDS)
        common = sorted(set(A) & set(B))
        if len(common) < 2:
            continue
        a = {s: A[s] for s in common}; b = {s: B[s] for s in common}
        for kind, label in [("mean", METRICS[4]), ("worst", METRICS[5]), ("high", METRICS[6])]:
            full = np.arange(N)
            obs = boot_metric(a, y, full, n, kind, hi) - boot_metric(b, y, full, n, kind, hi)
            diffs = np.empty(N_BOOT)
            for k in range(N_BOOT):
                idx = rng.integers(0, N, N); hidx = hi[rng.integers(0, len(hi), len(hi))] if len(hi) else hi
                diffs[k] = boot_metric(a, y, idx, n, kind, hidx) - boot_metric(b, y, idx, n, kind, hidx)
            p = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
            boot.append({"dataset": ds, "task": lab, "final": FINAL, "vs": base, "metric": label, "delta": obs,
                         "ci95_low": np.percentile(diffs, 2.5), "ci95_high": np.percentile(diffs, 97.5),
                         "p_value": p, "n_seeds": len(common)})
boot_df = pd.DataFrame(boot)
boot_df["result"] = np.where(boot_df["p_value"] < 0.05, np.where(boot_df["delta"] > 0, "better", "worse"), "n.s.")
display(boot_df.pivot_table(index=["dataset", "task", "vs"], columns="metric", values="delta", aggfunc="first").round(4))
boot_df.pivot_table(index=["dataset", "task", "vs"], columns="metric", values="result", aggfunc="first")

## 12. Part B — cross-dataset generalisation (binary harmful vs. not, OCR text, macro-F1)

In [ ]:
XD_MODELS = ["Fusion (mixed text)", "NAFIM", "NAFIM + image fallback"]
xrows = []
for m in XD_MODELS:
    for ds, lab in XD_TRAIN:
        for s in XD_SEEDS:
            tid = f"{ds}__{lab}__{slug(m)}__s{s}"
            # in-domain cell: OCR-text test set of the training dataset
            f_in = f"{PRED}/{tid}__test-ocr.npy"
            if os.path.exists(f_in):
                y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy")
                xrows.append({"model": m, "train_on": ds, "test_on": ds, "seed": s,
                              "f1": f1_score(y, np.load(f_in).argmax(1), average="macro")})
            for tgt in xd:
                f = f"{PRED}/{tid}__xd-{tgt}.npy"
                if tgt != ds and os.path.exists(f):
                    xrows.append({"model": m, "train_on": ds, "test_on": tgt, "seed": s,
                                  "f1": f1_score(xd[tgt]["y"], np.load(f).argmax(1), average="macro")})
xcmp = pd.DataFrame(xrows)
xtab = xcmp.pivot_table(index=["train_on", "model"], columns="test_on", values="f1", aggfunc=fmt)
xcmp["in_domain"] = xcmp["train_on"] == xcmp["test_on"]
summary = (xcmp[~xcmp.in_domain].groupby(["model", "seed"])["f1"].mean().groupby("model").agg(["mean", "std"]).round(4)
           .rename(columns={"mean": "mean off-diagonal F1", "std": "std over seeds"}))
display(summary)
xtab

## 13. Part C — ablations (Δ vs. full NAFIM on the same seeds; paired t-test)

In [ ]:
ABL_MODELS = ["NAFIM"] + ABLATIONS
abl = pd.DataFrame([r for ds, lab in TASKS for m in ABL_MODELS for r in metric_rows(m, ds, lab, ABL_SEEDS)])
KEY = [METRICS[4], METRICS[5], METRICS[6]]
arows = []
for (ds, lab), grp in abl.groupby(["dataset", "task"]):
    full = grp[grp.model == "NAFIM"].set_index("seed")
    for m in ABLATIONS:
        g2 = grp[grp.model == m].set_index("seed"); common = full.index.intersection(g2.index)
        if len(common) < 2:
            continue
        for k in KEY:
            d = g2.loc[common, k] - full.loc[common, k]
            arows.append({"dataset": ds, "task": lab, "ablation": m, "metric": k, "f1": g2.loc[common, k].mean(),
                          "delta_vs_full": d.mean(), "p_value": ttest_rel(g2.loc[common, k], full.loc[common, k]).pvalue})
abl_df = pd.DataFrame(arows, columns=["dataset", "task", "ablation", "metric", "f1", "delta_vs_full", "p_value"])
abl_tab = abl.groupby(["dataset", "task", "model"])[KEY].agg(fmt)
display(abl_tab)
abl_df.pivot_table(index=["dataset", "task", "ablation"], columns="metric", values="delta_vs_full", aggfunc="first").round(4)

## 14. Figures — gate behaviour (mean predicted reliability by OCR error level) and cross-dataset heatmap

In [ ]:
BKT = [("CER < 0.2", 0, 0.2), ("0.2 – 0.5", 0.2, 0.5), ("CER ≥ 0.5", 0.5, np.inf)]
fig, axes = plt.subplots(1, len(TASKS), figsize=(4.5 * len(TASKS), 3.6), sharey=True)
for ax, (ds, lab) in zip(np.atleast_1d(axes), TASKS):
    df = data[ds]["df"]; te = idx_of(df, "test")
    for src, col in [("surya", "C0"), ("easyocr", "C1")]:
        c = df[f"cer_{src}"].values[te]; means = []
        for _, lo, hi_ in BKT:
            m = (c >= lo) & (c < hi_); vals = []
            for s in SEEDS:
                f = f"{PRED04}/{ds}__{lab}__NAFIM__s{s}__test-{src}__rel.npy"
                if os.path.exists(f) and m.any():
                    vals.append(np.load(f)[m].mean())
            means.append(np.mean(vals) if vals else np.nan)
        ax.plot([b[0] for b in BKT], means, marker="o", color=col, label=f"{src} text")
    ax.set_title(f"{ds}/{lab}"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("mean predicted reliability r"); np.atleast_1d(axes)[0].legend(fontsize=8)
fig.suptitle("NAFIM gate: predicted text reliability falls as true OCR error rises (test set, 5 seeds)")
fig.tight_layout(); fig.savefig(f"{RES}/gate_by_cer.png", dpi=200); plt.show()

if len(xcmp):
    tgts = list(XD_TARGETS)
    fig, axes = plt.subplots(1, len(XD_MODELS), figsize=(5.2 * len(XD_MODELS), 3.6),
                             gridspec_kw={"wspace": 0.55})
    for ax, m in zip(np.atleast_1d(axes), XD_MODELS):
        M = np.full((len(XD_TRAIN), len(tgts)), np.nan)
        for i, (ds, _) in enumerate(XD_TRAIN):
            for j, t in enumerate(tgts):
                v_ = xcmp[(xcmp.model == m) & (xcmp.train_on == ds) & (xcmp.test_on == t)]["f1"]
                M[i, j] = v_.mean() if len(v_) else np.nan
        im = ax.imshow(M, vmin=0.4, vmax=0.85, cmap="viridis")
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                if not np.isnan(M[i, j]):
                    ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", color="w", fontsize=9)
        ax.set_xticks(range(len(tgts))); ax.set_xticklabels(tgts, rotation=20, fontsize=8)
        ax.set_yticks(range(len(XD_TRAIN))); ax.set_yticklabels([d for d, _ in XD_TRAIN], fontsize=8)
        ax.set_title(m, fontsize=9); ax.set_xlabel("test on")
    np.atleast_1d(axes)[0].set_ylabel("train on")
    fig.colorbar(im, ax=axes, shrink=0.8, label="macro-F1")
    fig.savefig(f"{RES}/cross_dataset_heatmap.png", dpi=200, bbox_inches="tight"); plt.show()

## 15. Save results + config

In [ ]:
val_tab.to_csv(f"{RES}/D_validation_selection.csv")
main_cmp.to_csv(f"{RES}/A_main_per_seed.csv", index=False); main_tab.to_csv(f"{RES}/A_main_table.csv")
boot_df.to_csv(f"{RES}/E_bootstrap_final_vs_baselines.csv", index=False)
xcmp.to_csv(f"{RES}/B_cross_dataset_per_seed.csv", index=False); xtab.to_csv(f"{RES}/B_cross_dataset_table.csv")
abl.to_csv(f"{RES}/C_ablation_per_seed.csv", index=False); abl_df.to_csv(f"{RES}/C_ablation_deltas.csv", index=False)
config = {
    "final_model": FINAL, "final_model_selection": "highest mean validation macro-F1 (gold/Surya/EasyOCR) over tasks and seeds",
    "variants": {k: {**{kk: vv for kk, vv in v.items() if kk != "mix"}, "mix": v["mix"]} for k, v in VARIANTS.items()},
    "lambda_img_aux": LAMBDA_IMG, "seeds": SEEDS, "ablation_seeds": ABL_SEEDS, "cross_dataset_seeds": XD_SEEDS,
    "cross_dataset": {"train": XD_TRAIN, "test_text": XD_TARGETS, "label": "label_bin, test split",
                      "quality_feature_standardisation": "statistics of the training dataset"},
    "bootstrap": {"n": N_BOOT, "unit": "test memes (paired, same resample for both models), metric averaged over seeds"},
    "notes": ["'NAFIM' and 'Fusion (mixed text)' main 5-seed results come from notebook 04; they are re-trained here only for cross-dataset tests",
              "'NAFIM w/o consistency' keeps the two-view (gold + noisy) cross-entropy and removes only the KL term"],
    "inherited": {"nb02_hyperparameters": cfg02, "encoder": TEXT_MODELS[ENC_KEY], "word_embeddings_frozen": FREEZE_WORD_EMB},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__,
                 "gpu": torch.cuda.get_device_name(0)},
}
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("saved to", RES)